# Referencia serial de Wiki Analytics — muestra local por cliente

Adaptación del notebook de Kaggle a los requisitos aclarados. Lee los Parquet locales y genera **Q1–Q5 para cada cliente por separado**, sin importar código Go. La ejecución de este notebook es un paso manual de preparación, nunca parte de la regresión automática.

- Q2: nombre, URL del artículo y bytes originales de imagen; ausencia explícita si no hay referencia.
- Q3: intersección de vocabularios por idioma, excluyendo stopwords y números.
- Q4: palabras distintas por abstract, top 20 con orden `(-count, word)`.
- Q5: ocurrencias, excluyendo stopwords y números; comparación exacta `count * N > S`.

Las listas NLTK inglesa/francesa están congeladas en `tests/oracle/resources/`. El sistema Go deberá usar esas mismas listas y el contrato de tokenización. Las imágenes pendientes o con HTTP 404 mantienen **Q2 y el paquete global incompletos**; no se reemplazan por resultados vacíos. Ver `tests/oracle/README.md`.

In [ ]:
from pathlib import Path
import os
import sys
import json

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "dataset").is_dir() and
               (path / "wikipedia-article-analysis.ipynb").is_file())
sys.path.insert(0, str(ROOT / "tests" / "oracle"))
import reference

CONFIG = reference.notebook_config(ROOT)
CLIENTS = CONFIG["clients"]
ROWS_PER_FILE = CONFIG["rows_per_file"]
OUTPUT = CONFIG["output"]
DATASET = CONFIG["dataset"]
IMAGE_MODE = CONFIG["image_mode"]
IMAGE_FILE = CONFIG["image_file"]
print({key: str(value) if isinstance(value, Path) else value
       for key, value in CONFIG.items()})


## Contrato semántico y stopwords

`reference.py` contiene las operaciones seriales del notebook original y el soporte de entrada/salida. Python usa `str.lower()`, `re.findall(r"\b\w+\b", ...)` en Unicode y el filtro numérico original. No se aplica stemming ni eliminación de acentos. Q4 convierte los tokens de cada abstract en un conjunto; Q5 conserva las repeticiones. Abstract histórico nulo se excluye del denominador; vacío no nulo aporta cero y sí cuenta. Sin base histórica, Q5 queda vacío con estado explícito.

In [ ]:
STOPWORDS = reference.load_stopwords()
SEMANTICS = reference.contract()
assert reference.tokens("the alpha alpha et beta 123", STOPWORDS) == ["alpha", "alpha", "beta"]
print(json.dumps(SEMANTICS, ensure_ascii=False, indent=2))

## Entrada y partición reproducible

Se intercalan archivos `en`, `fr` por índice numérico. La posición global empieza en cero y la fila se asigna mediante `ordinal % CLIENTS`. Los índices no dependen del tamaño del lote. Cada manifiesto guarda rutas, hashes, esquema y cantidades; los Parquet permanecen en su ubicación original. El lector del oráculo proyecta campos analíticos; el cliente del sistema deberá transmitir registros completos.

Cada ocurrencia conserva identidad `(archivo, índice de fila)`: no se deduplican artículos repetidos. El presupuesto de la muestra completa es 467.363.817 bytes comprimidos, cercano a los 500 MB sugeridos.

In [ ]:
INPUTS = reference.inventory(DATASET, CLIENTS, ROWS_PER_FILE)
print(json.dumps({"files": len(INPUTS["files"]),
                  "selected_rows": INPUTS["selected_rows"],
                  "source_parquet_bytes": INPUTS["source_parquet_bytes"],
                  "rows_per_client": [len(range(c, INPUTS["selected_rows"], CLIENTS))
                                      for c in range(CLIENTS)]}, indent=2))

## Ejecutar las cinco consultas, un cliente a la vez

El destino debe ser nuevo: no se sobrescriben referencias ya generadas. Para cada cliente se recorre la entrada por lotes, manteniendo vocabularios, conteos y candidatos de ese cliente. No se carga el corpus completo ni se combinan estados entre clientes. Q5 espera su suma y denominador históricos definitivos.

Se exportan todas las filas a JSONL, sin truncamiento de presentación. Q1/Q2/Q5 conservan multiplicidades, Q3 tiene palabras únicas y Q4 tiene orden total. Q2 registra referencias pendientes hasta capturar sus bytes.

In [ ]:
MANIFEST = reference.create_bundle(OUTPUT, INPUTS, ROOT / "wikipedia-article-analysis.ipynb")
for client in range(CLIENTS):
    print(f"Procesando cliente {client}...", flush=True)
    results, stats = reference.serial_queries(
        reference.client_rows(DATASET, INPUTS, client), STOPWORDS)
    entry = reference.export_client(OUTPUT, MANIFEST, client, results, stats)
    print(json.dumps({"client": entry["id"], "input_rows": entry["input_rows"],
                      "languages": entry["languages"],
                      "historical_sum": entry["historical_sum"],
                      "historical_count": entry["historical_count"],
                      "query_rows": {q: m["rows"] for q, m in entry["queries"].items()}},
                     ensure_ascii=False), flush=True)
    del results
reference.finish_bundle(OUTPUT, MANIFEST)

## Imágenes Q2: defer, network o fixture

El modo `defer` (default) conserva URLs como `pending`, sin red. `network` usa la captura HTTPS original de `image.content_url` y su caché, sin miniaturas, sustituciones ni fallback. Los HTTP 404 mantienen Q2 incompleta; otros errores detienen la preparación.

`fixture` usa `WIKI_ORACLE_IMAGE_FILE` para las referencias presentes, con procedencia sintética explícita; no agrega esas URLs a la caché de red. Artículos sin imagen conservan `None`. Los modos se seleccionan con `WIKI_ORACLE_IMAGE_MODE`; archivo, modo y cantidades se validan antes de crear el paquete. El antiguo flag `WIKI_ORACLE_DOWNLOAD_IMAGES` fue retirado.


In [ ]:
IMAGE_AUDIT = reference.handle_images(OUTPUT, IMAGE_MODE, IMAGE_FILE)
print(json.dumps(IMAGE_AUDIT, ensure_ascii=False, indent=2))


## Auditoría y estado final

La auditoría comprueba hashes y cantidades de los archivos exportados. No reemplaza la comparación contra el sistema distribuido. El comando `reference.py verify` exige por defecto todas las imágenes y devuelve error si faltan. Sólo `status == "complete"` habilita el paquete entero como referencia Q1–Q5.

In [ ]:
AUDIT = reference.verify_bundle(OUTPUT, DATASET, require_images=False)
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))
if AUDIT["status"] != "complete":
    print("ATENCIÓN: Q1/Q3/Q4/Q5 exportadas; Q2 pendiente de bytes. No validar Q2 como exitosa.")
else:
    print("Referencias Q1–Q5 completas. Conservar este paquete y sus imágenes.")